# 标签噪声与错误分析

**负责人：** 2 号　**阶段：** 4 分析

解释模型在哪里出错，以及其中有多少是标签本身的问题。

## 任务

标"（选做）"的任务在必做部分完成后再做。

- [x] 计算 VADER 和给定标签的一致率，检验"标签由 VADER 生成"这个假设（只做分析，绝不能当特征）
- [ ] 找出所有模型都分错的推文，判断是不是通用情感词典误读了金融术语

## 产出

- 报告 4.5 节用的表格和例子
- `figures/analysis_*.png`

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

# Part A: where do the labels come from?

The dataset description says the tweets were labelled Positive / Neutral / Negative, but not **how**. If the labels were produced by a sentiment lexicon rather than by people, three things follow:

1. our classifiers learn to imitate that tool, so its blind spots become our "ground truth";
2. the tool's score must **never** be used as a feature (it would leak the label);
3. disagreements between the labels and real financial meaning are a business insight, not just noise.

**Hypothesis:** the labels are VADER compound scores thresholded at ±0.05 (the threshold recommended by VADER's authors), computed on the original tweet text.

**Test:** if the hypothesis is true, VADER should reproduce the labels far better than an unrelated sentiment lexicon (TextBlob), work best on the raw text, and agree *more* on tweets VADER cannot read (non-English → score 0 → Neutral).

The test uses every readable, non-duplicate tweet from `clean_v2_all.csv`: the retained tweets plus those removed as non-English or word-less, because those removed groups are part of the evidence.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from src.config import CLEAN_DIR, CLEAN_VERSION

full = pd.read_csv(CLEAN_DIR / f"clean_v{CLEAN_VERSION}_all.csv", keep_default_na=False)
readable = full["keep"].eq(1) | full["drop_reason"].isin(["non_english", "non_english_manual", "no_content"])
df = full[readable].reset_index(drop=True)
sia = SentimentIntensityAnalyzer()


def to_label(score, threshold):
    """Map a polarity score to the dataset's codes: 1 Positive, 2 Negative, 0 Neutral."""
    return np.where(score >= threshold, 1, np.where(score <= -threshold, 2, 0))


df["vader_raw"] = df["text_raw"].map(lambda s: sia.polarity_scores(s)["compound"])
df["vader_clean"] = df["text_clean"].map(lambda s: sia.polarity_scores(s)["compound"])
df["textblob_raw"] = df["text_raw"].map(lambda s: TextBlob(s).sentiment.polarity)
print(f"{len(df)} readable tweets (clean_v{CLEAN_VERSION}): {int(df['keep'].sum())} retained + "
      f"{int((df['keep'] == 0).sum())} removed as non-English or word-less")

## A1. Does VADER reproduce the labels better than another sentiment tool?

In [ ]:
candidates = {
    "VADER, raw text, ±0.05": to_label(df["vader_raw"], 0.05),
    "VADER, raw text, any non-zero": to_label(df["vader_raw"], 1e-9),
    "VADER, cleaned text_clean, ±0.05": to_label(df["vader_clean"], 0.05),
    "TextBlob, raw text, any non-zero": to_label(df["textblob_raw"], 1e-9),
    "Majority class (always Positive)": np.ones(len(df), dtype=int),
}
agreement = pd.DataFrame([
    {"labeller": name, "agreement": (pred == df["label"]).mean(), "Cohen's kappa": cohen_kappa_score(df["label"], pred)}
    for name, pred in candidates.items()
]).round(3)
agreement

VADER on the raw text reproduces the given labels far better than TextBlob, a different lexicon, and the agreement is "almost perfect" by Cohen's kappa. Agreement **drops** on the cleaned text, mostly because restored contractions add negations the labeller never saw (see `data/clean_v2/clean_v2_log.md`).

## A2. Where does VADER disagree with the labels?

In [ ]:
df["vader_label"] = candidates["VADER, raw text, ±0.05"]
df["agree"] = df["vader_label"] == df["label"]


def group(row):
    if row["drop_reason"].startswith("non_english"):
        return "Non-English (removed)"
    if row["drop_reason"] == "no_content":
        return "No words (removed)"
    return "Too short to detect" if row["lang"] == "short" else "English"


by_lang = df.groupby(df.apply(group, axis=1))["agree"].agg(agreement="mean", tweets="size").round(3)
by_lost = df.groupby(np.where(df["emoji_lost"] == 1, "has lost characters", "no lost characters"))["agree"].agg(
    agreement="mean", tweets="size").round(3)
pd.concat({"language": by_lang, "characters lost in the source file": by_lost})

- **Non-English and word-less tweets agree most.** VADER only knows English words, so it scores them 0 and they get *Neutral*, which is what the labels say for nearly all of them. This is why `clean_v2` removes them: their labels say nothing about sentiment.
- **Tweets with lost characters agree less.** Emoji carry sentiment in VADER. If the labels were computed before the emoji were destroyed in the file we received, VADER on today's text cannot see them, so these tweets disagree more often.

In [ ]:
fig = plot_confusion(df["label"], df["vader_label"], "Given label vs VADER (raw text, ±0.05)",
                     save_as=FIGURES_DIR / "analysis_vader_vs_labels.png")

In [ ]:
# Disagreements among the retained tweets are candidates for label noise; 21_m2_knn compares them with its own flags
retained = df["keep"] == 1
disagree = df.loc[retained & ~df["agree"], ["id", "label", "vader_label", "vader_raw", "lang", "emoji_lost", "text_raw"]].copy()
disagree["label"] = disagree["label"].map(LABELS)
disagree["vader_label"] = disagree["vader_label"].map(LABELS)
disagree.to_csv(CLEAN_DIR / f"vader_disagreements_v{CLEAN_VERSION}.csv", index=False, encoding="utf-8")
print(f"{len(disagree)} disagreements among {int(retained.sum())} retained tweets saved")
disagree.sample(10, random_state=SEED)

## A3. What this means for the project

- **The hypothesis is strongly supported**, but it remains a hypothesis: we cannot see the original pipeline, and the remaining disagreements can come from emoji lost in the file, restored contractions, a different VADER version or preprocessing we cannot reproduce.
- **The task is effectively "imitate VADER".** Models that rely on sentiment-bearing words have an advantage; semantic models such as SBERT may "disagree" with the labels when they understand finance better than VADER (*bearish*, *crash*, *short*).
- **VADER scores must never be a model input.** They are used here only to analyse the labels.
- **Business insight to develop in Part B:** where the models and VADER both misread financial language, a generic sentiment tool would give a misleading signal to an investor dashboard.

# Part B: model error analysis

To do after all six models have saved out-of-fold predictions (`src.cache.load_preds`):

- [ ] tweets every model gets wrong, and how many of them are VADER disagreements from A2
- [ ] whether finance terms (bear, bull, crash, dip, short, puts, red, green) are over-represented in those errors
- [ ] whether removing the A2 disagreements from training changes macro-F1